# Data Engineering
---

In [1]:
import pandas as pd
import json
from pymongo import MongoClient
from collections import defaultdict
import re

## 1 - Data Loading

In [2]:
# connect to MongoDB and initialize client 
client = MongoClient('mongodb://localhost:27017/')
db = client['novacred_db']
collection = db['applications'] # define collection for more reliability  

# drop collection before loading multiple times
collection.drop()

# load JSON file
with open('../data/raw_credit_applications.json', 'r') as file:
    raw_data = json.load(file)

## 2 - Data Cleaning 

Following the project guidelines, we will address the six core data quality issues:
### 1. Duplicate records
MongoDB requires unique _id fields. Therefore, we check for duplicates.

In [3]:
unique_records = {}
duplicates_count = 0
# as "_id" is a unique identifier, documents with the same _id shall be deleted 
for application in raw_data:
    app_id = application['_id']
    
    # count duplicates
    if app_id in unique_records:
        duplicates_count += 1
        print(f"Duplicate found: {app_id}")
    
    # duplicates will be removed through overwriting
    unique_records[app_id] = application

clean_data = list(unique_records.values())

print(f"Found and removed {duplicates_count} duplicates.")

Duplicate found: app_042
Duplicate found: app_001
Found and removed 2 duplicates.


The duplicated documents are resubmitted applications with duplicate IDs. We will kept the most recent resubmission and load the unique records into the database.

In [4]:
# loading cleaned data 
collection.insert_many(clean_data)
print(f"Inserted {len(clean_data)} unique documents into MongoDB.")

Inserted 500 unique documents into MongoDB.


### 2. Inconsistent data types
Instead of guessing which fields have inconsistent data types, we will dynamically audit the entire collection. 
This script flattens every document and records the Python data types found for each field. If a field contains more than one data type across the dataset (e.g., both integers and strings), it will be flagged for cleaning.

In [5]:
# Dictionary to hold sets of data types for every field
field_types = defaultdict(set)

# Helper function to flatten nested JSON (e.g., turns {'financials': {'income': 50}} into 'financials.income': 50)
def flatten_doc(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_doc(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # For this audit, we skip complex arrays like spending_behavior
            pass 
        else:
            # We ignore null/None values for the type consistency check
            if v is not None and v != "":
                items.append((new_key, type(v).__name__))
    return dict(items)

# Scan every document in the database
for doc in collection.find({}):
    doc.pop('_id', None) # We don't need to check the MongoDB ID
    flat_doc = flatten_doc(doc)
    
    for field, data_type in flat_doc.items():
        field_types[field].add(data_type)

inconsistent_fields = {field: types for field, types in field_types.items() if len(types) > 1}

if not inconsistent_fields:
    print("No inconsistent data types found.")
else:
    for field, types in inconsistent_fields.items():
        print(f"FLAGGED: '{field}' contains mixed types: {types}")

FLAGGED: 'financials.annual_income' contains mixed types: {'float', 'str', 'int'}


In [6]:
# audit flagged 'financials.annual_income' as having mixed string/int types
# We will dynamically find any strings in these numeric fields and cast them to integers

fields_to_fix = list(inconsistent_fields.keys())

for field in fields_to_fix:
    # Update query: Find documents where this field is a string, and cast it to an int
    result = collection.update_many(
        {field: {"$type": "string"}},
        [{"$set": {field: {"$toInt": f"${field}"}}}]
    )
    if result.modified_count > 0:
        print(f"Fixed: Casted {result.modified_count} string values to integers in '{field}'.")

Fixed: Casted 8 string values to integers in 'financials.annual_income'.


### 3. Missing or incomplete records
Before altering the database, we must audit **all** fields for missing values (nulls, empty strings, or completely missing keys). 

**Business Rules for Handling Missing Data:**
When missing data is found, we apply the following logic based on the field's importance to the credit application:
1. **Critical Identifiers (e.g., SSN):** A credit application is legally invalid without an SSN. If this is missing, the entire record must be **dropped**.
2. **Contact Information (e.g., Email):** Non-critical for the credit risk algorithm. If missing, we will **impute** it with a placeholder (e.g., "UNKNOWN") to preserve the rest of the valid financial data.
3. **Financial/Algorithmic Data:** If critical numeric fields used for bias analysis (like income) are missing, they must be evaluated for dropping or median imputation.

In [7]:
# 1. First, we dynamically find EVERY possible field name in our dataset
all_fields = set()
for doc in collection.find({}):
    doc.pop('_id', None) # Ignore the MongoDB ID
    # We reuse the flatten_doc function from Step 2 to get all nested keys
    all_fields.update(flatten_doc(doc).keys())

# 2. Now, we query MongoDB to count missing values for every single field
missing_report = {}

for field in all_fields:
    missing_count = collection.count_documents({
        "$or": [
            {field: {"$exists": False}},
            {field: None},
            {field: ""}
        ]
    })
    if missing_count > 0:
        missing_report[field] = missing_count

print("--- COMPREHENSIVE MISSING DATA AUDIT ---")
if not missing_report:
    print("No missing data found in any field.")
else:
    for field, count in sorted(missing_report.items()):
        print(f"FLAGGED: '{field}' is missing or empty in {count} records.")

--- COMPREHENSIVE MISSING DATA AUDIT ---
FLAGGED: 'applicant_info.date_of_birth' is missing or empty in 5 records.
FLAGGED: 'applicant_info.email' is missing or empty in 7 records.
FLAGGED: 'applicant_info.gender' is missing or empty in 3 records.
FLAGGED: 'applicant_info.ip_address' is missing or empty in 5 records.
FLAGGED: 'applicant_info.ssn' is missing or empty in 5 records.
FLAGGED: 'applicant_info.zip_code' is missing or empty in 2 records.
FLAGGED: 'decision.approved_amount' is missing or empty in 208 records.
FLAGGED: 'decision.interest_rate' is missing or empty in 208 records.
FLAGGED: 'decision.rejection_reason' is missing or empty in 292 records.
FLAGGED: 'financials.annual_income' is missing or empty in 5 records.
FLAGGED: 'financials.annual_salary' is missing or empty in 495 records.
FLAGGED: 'loan_purpose' is missing or empty in 450 records.
FLAGGED: 'notes' is missing or empty in 498 records.
FLAGGED: 'processing_timestamp' is missing or empty in 438 records.


**Executing the Missing Data Fixes:**
Based on our comprehensive audit, we have discovered patterns in the missing data. We will apply the following business rules:
1. **Drop:** Records missing `ssn` are structurally invalid.
2. **Impute:** Categorical and demographic fields (`gender`, `email`, `zip_code`) will be filled with an "UNKNOWN" placeholder.
3. **Keep as Null:** Conditional fields (like `approved_amount` vs `rejection_reason`) and sparse fields (like `notes`) will be left alone. These will later naturally convert to `NaN` in Pandas.
4. **Rename (Schema Drift):** Our audit flagged 5 records as missing `annual_income`. However, further inspection revealed these records contain this exact financial data under an inconsistent key (`annual_salary`). We will rename this key to `annual_income` to perfectly align with the official data dictionary and resolve the "missing" data without dropping the rows.

In [8]:
# 1. DROP the 5 records missing an SSN
deleted = collection.delete_many({
    "$or": [{"applicant_info.ssn": {"$exists": False}}, {"applicant_info.ssn": None}, {"applicant_info.ssn": ""}]
})
print(f"Dropped {deleted.deleted_count} completely invalid records (Missing SSN).")

# 2. IMPUTE "UNKNOWN" for demographics and contact info
# Fix Emails
updated_emails = collection.update_many(
    {"$or": [{"applicant_info.email": {"$exists": False}}, {"applicant_info.email": None}, {"applicant_info.email": ""}]},
    {"$set": {"applicant_info.email": "UNKNOWN_EMAIL"}}
)
print(f"Imputed {updated_emails.modified_count} missing emails.")

# Fix Gender (Crucial for Bias Analysis)
updated_genders = collection.update_many(
    {"$or": [{"applicant_info.gender": {"$exists": False}}, {"applicant_info.gender": None}, {"applicant_info.gender": ""}]},
    {"$set": {"applicant_info.gender": "UNKNOWN"}}
)
print(f"Imputed {updated_genders.modified_count} missing genders.")

# Fix Zip Codes (Crucial for Redlining Analysis)
updated_zips = collection.update_many(
    {"$or": [{"applicant_info.zip_code": {"$exists": False}}, {"applicant_info.zip_code": None}, {"applicant_info.zip_code": ""}]},
    {"$set": {"applicant_info.zip_code": "UNKNOWN_ZIP"}}
)
print(f"Imputed {updated_zips.modified_count} missing zip codes.")

# Fix Schema Drift: Rename 'annual_salary' to resolve the 5 "missing" annual_income records
renamed_fields = collection.update_many(
    {"financials.annual_salary": {"$exists": True}},
    {"$rename": {"financials.annual_salary": "financials.annual_income"}}
)
print(f"Renamed {renamed_fields.modified_count} 'annual_salary' fields to 'annual_income'.")

Dropped 5 completely invalid records (Missing SSN).
Imputed 3 missing emails.
Imputed 0 missing genders.
Imputed 0 missing zip codes.
Renamed 5 'annual_salary' fields to 'annual_income'.


Based on the audit results, the 5 records missing an SSN were completely blank "ghost" applications that also accounted for the missing demographic fields (such as gender and zip code). Therefore, dropping these 5 structurally invalid applications simultaneously resolved the missing data issues across those other fields, leaving a clean dataset for the Data Scientist.

### 4. Inconsistent coding/formatting of categorical fields
To ensure our categorical variables are formatted consistently, we first audit the database to extract every unique value present in these fields. Once we identify the inconsistent variations (e.g., abbreviations, varying capitalization), we will apply a strict mapping logic to standardize them.

In [9]:
# Define the known categorical fields
categorical_fields = [
    "applicant_info.gender", 
    "loan_purpose", 
    "decision.rejection_reason"
]

for field in categorical_fields:
    # Use MongoDB's distinct() to grab all unique values currently in the database
    unique_values = collection.distinct(field)
    
    # Filter out None/UNKNOWN just to see the actual messy data
    clean_unique_values = [v for v in unique_values if v not in [None, "UNKNOWN"]]
    
    print(f"\nUnique values found in '{field}':")
    print(clean_unique_values)


Unique values found in 'applicant_info.gender':
['F', 'Female', 'M', 'Male']

Unique values found in 'loan_purpose':
['auto', 'business', 'debt_consolidation', 'education', 'home_improvement', 'medical', 'moving', 'personal', 'vacation', 'wedding']

Unique values found in 'decision.rejection_reason':
['algorithm_risk_score', 'high_dti_ratio', 'insufficient_credit_history', 'low_income']


**Executing the Categorical Standardization:**
Based on the audit results, the `applicant_info.gender` field contains inconsistent coding (mixing "M"/"Male" and "F"/"Female"). We will standardize these to use the full words "Male" and "Female" to ensure accurate demographic groupings for the downstream Bias Analysis.

In [10]:
# We only need to target the specific abbreviations found in our audit
gender_fixes = {
    "M": "Male",
    "F": "Female"
}

total_fixed = 0

for messy_value, standard_value in gender_fixes.items():
    # Find any document with the abbreviated gender and replace it with the full word
    updated_gender = collection.update_many(
        {"applicant_info.gender": messy_value},
        {"$set": {"applicant_info.gender": standard_value}}
    )
    
    if updated_gender.modified_count > 0:
        print(f"Standardized {updated_gender.modified_count} records from '{messy_value}' to '{standard_value}'.")
        total_fixed += updated_gender.modified_count

if total_fixed == 0:
    print("No categorical formatting fixes were needed.")
else:
    print(f"\nFixed a total of {total_fixed} records.")

Standardized 53 records from 'M' to 'Male'.
Standardized 58 records from 'F' to 'Female'.

Fixed a total of 111 records.


### 5. Invalid or impossible values
In the context of a credit application, financial figures (like income or debt) and time-based figures (like credit history duration) cannot logically be less than zero. We will dynamically audit the entire database to find any numeric field that contains negative values.

In [11]:
# Helper function to get the actual value from a nested dictionary using a dot-notation string
def get_actual_value(document, dot_notation_key):
    keys = dot_notation_key.split('.')
    current_val = document
    for key in keys:
        if isinstance(current_val, dict):
            current_val = current_val.get(key)
        else:
            return None
    return current_val

# 1. Ensure a list of all fields. 
all_fields = set()
for doc in collection.find({}):
    doc_copy = dict(doc)
    doc_copy.pop('_id', None)
    # Re-using your existing flatten_doc just to grab the keys
    all_fields.update(flatten_doc(doc_copy).keys())

impossible_report = {}

# 2. Dynamically scan all fields for negative numbers and store the documents
for field in all_fields:
    query = {
        "$and": [
            {field: {"$type": ["int", "double", "long", "decimal"]}},
            {field: {"$lt": 0}}
        ]
    }
    
    # Fetch the actual broken documents
    bad_docs = list(collection.find(query))
    
    if len(bad_docs) > 0:
        impossible_report[field] = bad_docs

print("--- IMPOSSIBLE VALUES AUDIT ---")
if not impossible_report:
    print("No negative or impossible numeric values found.")
else:
    for field, docs in sorted(impossible_report.items()):
        print(f"\nFLAGGED: '{field}' contains {len(docs)} records with negative values.")
        print("Affected documents:")
        
        for doc in docs:
            # Use our new helper function to grab the exact real number!
            bad_value = get_actual_value(doc, field)
            
            print(f"   - Application ID: {doc.get('_id')} | {field} = {bad_value}")

--- IMPOSSIBLE VALUES AUDIT ---

FLAGGED: 'financials.credit_history_months' contains 2 records with negative values.
Affected documents:
   - Application ID: app_043 | financials.credit_history_months = -10
   - Application ID: app_156 | financials.credit_history_months = -3

FLAGGED: 'financials.savings_balance' contains 1 records with negative values.
Affected documents:
   - Application ID: app_290 | financials.savings_balance = -5000


**Executing the Impossible Values Fix:**
Based on our audit, we identified negative values in both time-based and financial fields. We will apply strict governance rules to correct these:

1. **Time Metrics (`credit_history_months`):** Time cannot be negative. This is assumed to be a typographical error (a stray dash during entry). We will convert these to their absolute (positive) values.
2. **Financial Metrics (`savings_balance`):** A negative savings balance implies an overdrawn account or system error. Converting this to a positive number would artificially inflate the applicant's wealth. Instead, we will conservatively cap negative savings balances at `0` for the risk analysis.

In [12]:
# Rule 1: Fix Time Metrics (Absolute Value)
corrected_time = collection.update_many(
    {"financials.credit_history_months": {"$lt": 0}},
    [{"$set": {"financials.credit_history_months": {"$abs": "$financials.credit_history_months"}}}]
)
if corrected_time.modified_count > 0:
    print(f"Rule 1: Converted {corrected_time.modified_count} negative credit history months to positive.")

# Rule 2: Fix Financial Metrics (Floor at 0)
corrected_finance = collection.update_many(
    {"financials.savings_balance": {"$lt": 0}},
    {"$set": {"financials.savings_balance": 0}}
)
if corrected_finance.modified_count > 0:
    print(f"Rule 2: Capped {corrected_finance.modified_count} negative savings balances at $0.")

print("\nImpossible values correction complete.")

Rule 1: Converted 2 negative credit history months to positive.
Rule 2: Capped 1 negative savings balances at $0.

Impossible values correction complete.


### 6. Inconsistent date formats
Date and time data must be formatted to the ISO 8601 standard (`YYYY-MM-DD` for dates, and `YYYY-MM-DDTHH:MM:SS` for timestamps).
We will audit `applicant_info.date_of_birth` and `processing_timestamp` by flagging any record that deviates from these exact standard regex patterns, allowing us to inspect the messy data before standardizing it.

In [13]:
# We define the strict ISO patterns we EXPECT to see
date_standards = {
    "applicant_info.date_of_birth": r"^\d{4}-\d{2}-\d{2}$",  # Expected: YYYY-MM-DD
    "processing_timestamp": r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"  # Expected: YYYY-MM-DDTHH:MM:SS
}

inconsistent_dates_found = False

for field, pattern in date_standards.items():
    # Query: Field exists, is a string, but DOES NOT match our strict pattern
    query = {
        field: {"$exists": True, "$type": "string", "$not": {"$regex": pattern}}
    }
    
    total_bad = collection.count_documents(query)
    
    if total_bad > 0:
        inconsistent_dates_found = True
        print(f"\nFLAGGED: '{field}' contains {total_bad} records with non-standard formats.")
        
        # Grab a small sample of the bad documents to see what the mess looks like
        bad_sample = list(collection.find(query).limit(5))
        
        print("Sample of invalid formats found:")
        unique_samples = set()
        for doc in bad_sample:
            # Using our helper function from Step 5 to safely get the nested value
            val = get_actual_value(doc, field)
            unique_samples.add(val)
            
        for sample in unique_samples:
            print(f"   - {sample}")
            
    else:
        print(f"\n '{field}' is perfectly formatted.")

if not inconsistent_dates_found:
    print("\nNo date formatting issues found.")


FLAGGED: 'applicant_info.date_of_birth' contains 158 records with non-standard formats.
Sample of invalid formats found:
   - 14/02/1982
   - 01/12/1978
   - 28/01/1990
   - 18/07/1979
   - 1990/07/26

 'processing_timestamp' is perfectly formatted.


**Executing the Date Format Fix:**
Based on the audit, the non-standard date formats are using slashes (`YYYY/MM/DD`) instead of the ISO standard hyphens (`YYYY-MM-DD`). We will use MongoDB's `$replaceAll` operator to standardize these strings, ensuring downstream time-series and age-calculation algorithms do not fail.

In [14]:
# Target the date_of_birth field and replace all "/" with "-"
fixed_dates = collection.update_many(
    {"applicant_info.date_of_birth": {"$regex": "/"}},
    [{"$set": {
        "applicant_info.date_of_birth": {
            "$replaceAll": { "input": "$applicant_info.date_of_birth", "find": "/", "replacement": "-" }
        }
    }}]
)

if fixed_dates.modified_count > 0:
    print(f"Standardized {fixed_dates.modified_count} date formats to ISO 8601 (hyphens).")
else:
    print("No date formats needed fixing.")

Standardized 157 date formats to ISO 8601 (hyphens).


## 3 - Schema Validation

**Governance Rules Enforced:**
- All applications must contain the `applicant_info` and `financials` objects.
- Critical identifiers (`ssn`) and demographic data (`gender`, `email`) must be present and typed as strings.
- Financial metrics (`annual_income`, `credit_history_months`, `savings_balance`) must strictly be numeric types.
- Time and savings metrics cannot be logically negative (minimum value of 0).

In [15]:
# 1. Define the strict rules based on the official Data Dictionary
validation_schema = {
    "bsonType": "object",
    # We require the main structural objects to exist
    "required": ["applicant_info", "financials", "decision"],
    "properties": {
        "applicant_info": {
            "bsonType": "object",
            # Since our cleaning pipeline imputed missing values, ALL of these must now exist!
            "required": ["full_name", "email", "ssn", "ip_address", "gender", "date_of_birth", "zip_code"],
            "properties": {
                "full_name": {"bsonType": "string"},
                "email": {"bsonType": "string"},
                "ssn": {"bsonType": "string"},
                "ip_address": {"bsonType": "string"},
                "gender": {"bsonType": "string"},
                "date_of_birth": {"bsonType": "string"},
                "zip_code": {"bsonType": "string"}
            }
        },
        "financials": {
            "bsonType": "object",
            "required": ["annual_income", "credit_history_months", "debt_to_income", "savings_balance"],
            "properties": {
                "annual_income": {
                    "bsonType": ["int", "double", "long", "decimal"]
                },
                "credit_history_months": {
                    "bsonType": ["int", "long"], # Strictly Integer as per dictionary
                    "minimum": 0
                },
                "debt_to_income": {
                    "bsonType": ["int", "double", "long", "decimal"]
                },
                "savings_balance": {
                    "bsonType": ["int", "double", "long", "decimal"],
                    "minimum": 0
                }
            }
        },
        "spending_behavior": {
            "bsonType": ["array", "null"], # Can be an array of objects
            "items": {
                "bsonType": "object",
                "required": ["category", "amount"],
                "properties": {
                    "category": {"bsonType": "string"},
                    "amount": {"bsonType": ["int", "double", "long", "decimal"]}
                }
            }
        },
        "decision": {
            "bsonType": "object",
            "required": ["loan_approved"],
            "properties": {
                "loan_approved": {"bsonType": "bool"},
                "interest_rate": {"bsonType": ["int", "double", "long", "decimal"]},
                "approved_amount": {"bsonType": ["int", "double", "long", "decimal"]},
                "rejection_reason": {"bsonType": "string"}
            }
        }
    }
}

# 2. Lock the database for the future
db.command({
    "collMod": "applications",
    "validator": {"$jsonSchema": validation_schema},
    "validationLevel": "strict",
    "validationAction": "error"
})
print("Database locked: Comprehensive data dictionary schema applied.")

# 3. THE ULTIMATE TEST: Check the existing data against the rules
invalid_docs_count = collection.count_documents({
    "$nor": [{"$jsonSchema": validation_schema}]
})


if invalid_docs_count == 0:
    print("0 invalid documents found.")
else:
    print(f"ERROR: Found {invalid_docs_count} documents that still violate the schema.")
    
    bad_docs = collection.find({"$nor": [{"$jsonSchema": validation_schema}]}).limit(5)
    print("Sample of failing documents:")
    for doc in bad_docs:
        print(f"   -> Application ID: {doc.get('_id')}")

Database locked: Comprehensive data dictionary schema applied.
0 invalid documents found.


## 4 - Export and flatten cleaned Data

With the database locked and validated, the final step in the Data Engineering pipeline is to extract the clean data. We will export the data into two formats:
1. **JSON:** A clean, nested JSON file for application developers and NoSQL backups.
2. **CSV:** A flattened, tabular CSV file for the Data Science team.

In [16]:
# 1. Fetch all clean records from MongoDB into a list
data_list = list(collection.find({}))

# 2. Export to JSON (Preserves the nested structure)
with open('../data/clean_credit_applications.json', 'w') as json_file:
    json.dump(data_list, json_file, indent=4)

# 3. Use pd.json_normalize to flatten the nested JSON (e.g., applicant_info.gender)
clean_df = pd.json_normalize(data_list)

# 4. Export to CSV (Flattened tabular structure)
clean_df.to_csv('../data/clean_credit_applications.csv', index=False)


print("Exported nested clean data to JSON: data/clean_credit_applications.json")
print("Exported flattened clean data to CSV: data/clean_credit_applications.csv")
print(f"Total clean records ready for analysis: {len(clean_df)}")


Exported nested clean data to JSON: data/clean_credit_applications.json
Exported flattened clean data to CSV: data/clean_credit_applications.csv
Total clean records ready for analysis: 495
